In [16]:
import requests
from bs4 import BeautifulSoup
from bs4.element import Tag
from typing import Optional, List, Dict
import urllib.parse
import json
import datetime
import pandas as pd
from tqdm import tqdm

In [3]:
GATHERED_URL_FILE = r'D:\LLM\techblog-summarize\data\gathered_urls\DATA-gathered_article_urls.csv'

article_links = pd.read_csv(GATHERED_URL_FILE)
article_links.head()

,name,link,tags
0,VIBLO MOBILE APP CHÍNH THỨC RA MẮT – TRẢI NGHI...,https://viblo.asia/announcements/viblo-mobile-...,[]
1,Padding trong struct,https://viblo.asia/p/padding-trong-struct-kY4g...,['C/Cpp']
2,"Saudi Arabia AI: Understanding Its Types, and ...",https://viblo.asia/p/saudi-arabia-ai-understan...,"['Artificial Intelligence', 'Saudi Arabia Ai']"
3,Hướng dẫn tạo TLS Cetificiate Fullchain Self-s...,https://viblo.asia/p/huong-dan-tao-tls-cetific...,['Cài đặt SSL']
4,Tạo component React Markdown Preview và Publis...,https://viblo.asia/p/tao-component-react-markd...,"['Markdown', 'npm']"


In [ ]:
def crawl_article(url: str, verbose: bool = True) -> Optional[str]:
    if verbose:
        print(f'Crawling article: {url}')

    response = requests.get(url)

    if response.status_code != 200:
        raise Exception(f"Failed to fetch article. Status code: {response.status_code}")

    if verbose:
        print("Article fetched successfully.")

    # Parse the HTML content
    soup = BeautifulSoup(response.content, 'html.parser')

    ARTICLE_CLASS = 'article-content__body my-2 flex-fill'
    article_body = soup.find('div', class_=ARTICLE_CLASS)

    if article_body is None:
        if verbose:
            print("Article body not found.")
        return

    markdown = parse_html_to_md(article_body)
    return markdown


def parse_html_to_md(article_body: Tag) -> str:
    """
    Convert the HTML content of an article body to Markdown format.
    Handles headings, paragraphs, code blocks, and lists.

    Args:
        article_body (Tag): BeautifulSoup Tag object representing the article body.
    Returns:
        str: The article content in Markdown format.
    """

    if article_body is None:
        return ""

    # Safely get the first non-recursive child (if any)
    children_root = article_body.find_all(recursive=False)
    if not children_root:
        return ""
    article_body = children_root[0]

    markdown_content = ""
    # Iterate through the children of article_body and convert to markdown
    for child in article_body.children:
        # Ensure we only access .name on Tag instances to avoid attribute errors
        if not isinstance(child, Tag):
            text = str(child).strip()
            if text:
                markdown_content += f"{text}\n\n"
            continue

        # H1 -> # , H2 -> ## , P -> normal text, etc.
        if child.name == 'h1':
            markdown_content += f"# {child.get_text()}\n\n"
        elif child.name == 'h2':
            markdown_content += f"## {child.get_text()}\n\n"
        elif child.name == 'p':
            markdown_content += f"{child.get_text()}\n\n"
        elif child.name == 'pre':
            # try to get a nested <code> tag to determine language
            code_tag = child.find('code')
            code_text = code_tag.get_text() if code_tag else child.get_text()

            lang = ''
            if code_tag and code_tag.has_attr('class'):
                for cls in code_tag.get('class', []):
                    if cls.startswith('language-'):
                        lang = cls.split('language-')[-1]
                        break
                    if cls.startswith('lang-'):
                        lang = cls.split('lang-')[-1]
                        break
                    # common class names
                    if cls in ('python', 'py', 'javascript', 'js', 'bash', 'sh', 'html', 'css', 'json'):
                        lang = cls
                        break
            markdown_content += f"```{lang}\n{code_text}\n```\n\n"

        elif child.name == 'ul':
            for li in child.find_all('li'):
                markdown_content += f"- {li.get_text()}\n"
            markdown_content += "\n"
        elif child.name == 'ol':
            for idx, li in enumerate(child.find_all('li'), start=1):
                markdown_content += f"{idx}. {li.get_text()}\n"
            markdown_content += "\n"
        else:
            # Just add text for other tags
            markdown_content += f"{child.get_text()}\n\n"
    # Delete extra newlines
    markdown_content = '\n'.join([line for line in markdown_content.split('\n') if line.strip() != ''])
    return markdown_content


def bulk_crawl_articles(urls: List[str], verbose: bool = False) -> List[str]:
    """
    Crawl multiple articles given their URLs.

    Args:
        urls (List[str]): List of article URLs to crawl.

    Returns:
        List[str]: List of article contents in Markdown format corresponding to the input URLs.
    """
    articles = []
    for url in tqdm(urls, desc="Crawling articles"):
        try:
            markdown = crawl_article(url, verbose=verbose)
            articles.append(markdown)
        except Exception as e:
            print(f"Error crawling {url}: {e}")
            articles.append(None)
    return articles

article_mds = bulk_crawl_articles(article_links['link'].tolist(), verbose=False)


Crawling articles: 100%|██████████| 299/299 [04:58<00:00,  1.00it/s]


In [23]:
# Attach article_mds to the existing DataFrame as a new column
if len(article_mds) != len(article_links):
    raise ValueError(f"Length mismatch: article_mds={len(article_mds)} vs article_links={len(article_links)}")

article_links_md = article_links.copy()
article_links_md['markdown'] = article_mds

# Preview and (optionally) save
article_links_md.head()

ALL_CRAWLED_DATA_DIR = r'D:\LLM\techblog-summarize\data\all_crawled_data'
article_links_md.to_csv(f'{ALL_CRAWLED_DATA_DIR}/DATA-all_crawled_articles_with_md_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.csv', index=False)

# Uncomment to save to disk
# OUT_FILE = r'D:\LLM\techblog-summarize\data\gathered_urls\DATA-gathered_article_urls_with_md.csv'
# article_links_md.to_csv(OUT_FILE, index=False)
# print(f"Saved augmented dataframe to: {OUT_FILE}")

In [24]:
# Check for exact duplicated markdown entries in article_links_md['markdown']
md = article_links_md['markdown']
total = len(md)
nonnull_mask = md.notnull()
total_nonnull = nonnull_mask.sum()
unique_nonnull = md[nonnull_mask].nunique()
duplicate_count = total_nonnull - unique_nonnull

print(f"Total rows: {total}")
print(f"Markdown non-null: {total_nonnull}")
print(f"Unique markdown (non-null): {unique_nonnull}")
print(f"Number of duplicated markdown entries (non-null): {duplicate_count}")

dup_mask = md.duplicated(keep=False) & nonnull_mask

if not dup_mask.any():
    print("No duplicated markdown found.")
else:
    dup_df = article_links_md[dup_mask].copy()
    groups = dup_df.groupby('markdown').apply(lambda df: df.index.tolist()).to_dict()
    print(f"Found {len(groups)} duplicated markdown group(s). Showing details:")
    for i, (content, idxs) in enumerate(groups.items(), start=1):
        print(f"\nGroup {i}: {len(idxs)} duplicates at indices {idxs}")
        sample = article_links_md.loc[idxs[0], ['name', 'link']]
        print(f"  Example row {idxs[0]}: title={sample['name']!r}, link={sample['link']}")
        preview = content.replace('\n', ' ')
        if len(preview) > 200:
            preview = preview[:200] + "..."
        print(f"  Preview: {preview}")

    # Save duplicate rows for inspection
    REPORT = f"{ALL_CRAWLED_DATA_DIR}/duplicate_markdown_report_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv"
    dup_df.to_csv(REPORT, index=False)
    print(f"\nSaved duplicate rows to: {REPORT}")

Total rows: 299
Markdown non-null: 297
Unique markdown (non-null): 297
Number of duplicated markdown entries (non-null): 0
No duplicated markdown found.
